# Inicio y conexion con SQLITE

In [ ]:
# importaciones generales para todo el notebook
import sqlite3
import requests
from bs4 import BeautifulSoup
import time
import re

# conexion y creacion del entorno sqlite
conn = sqlite3.connect("books_pipeline.db")
cur = conn.cursor()

# activar clave foranea manualmente en sqlite
cur.execute("PRAGMA foreign_keys = ON")
print("Conexion exitosa")
print("Fase 1 completada")

Conexion exitosa
Fase 1 completada


# Extraccion de categorias

In [ ]:
# extraccion base de categorias
base_url = "https://books.toscrape.com/"
response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")

# capturar la lista del panel lateral
lista_categorias = soup.find("ul", class_="nav nav-list").find("li").find_all("li")

categorias_data = []
for li in lista_categorias:
    nombre = li.text.strip()
    link = base_url + li.a["href"]
    categorias_data.append((nombre, link))

# insertar categorias en la base de datos evitando duplicados
cur.executemany("INSERT OR IGNORE INTO categories (name) VALUES (?)", 
                [(c[0],) for c in categorias_data])
conn.commit()

# crear cache para poder mapear la categoria luego y no tener que consultar 1000 veces
cur.execute("SELECT id, name, FROM categories") #forma rapida de obtener estructura
categorias_db = {row[1]: row[0] for row in cur.fetchall()}

print(f"Fase 2 completada: {len(categorias_db)} categorias mapeadas y guardadas.")

Fase 2 completada: 50 categorias mapeadas y guardadas.


# Extracción de libros

In [ ]:
# mapeo estandar de estrellas a enteros
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
libros_totales = []

# funcion de extraccion
def scrapear_categoria(nombre_cat, url_cat, cat_id):
    libros_extraidos = []
    url_actual = url_cat
    
    while url_actual:
        res = requests.get(url_actual)
        soup_cat = BeautifulSoup(res.text, "html.parser")
        articulos = soup_cat.find_all("article", class_="product_pod")
        
        for art in articulos:
            titulo = art.h3.a["title"].strip()
            precio_txt = art.find("p", class_="price_color").text
            precio = float(precio_txt.replace("£", "").replace("Â", "").strip())
            clase_rating = art.find("p", class_="star-rating")["class"][1]
            rating = rating_map.get(clase_rating, 0)
            
            #armar la url del libro
            link_relativo = art.h3.a["href"].replace("../../../", "")
            url_libro = f"https://books.toscrape.com/catalogue/{link_relativo}"
            
            libros_extraidos.append((titulo, precio, rating, cat_id, url_libro))
            
        # buscar paginacion dentro de la misma categoria
        next_btn = soup_cat.find("li", class_="next")
        if next_btn:
            url_actual = url_actual.rsplit("/", 1)[0] + "/" + next_btn.a["href"]
        else:
            url_actual = None
            
    return libros_extraidos

print("iniciando extraccion secuencial de categorias...")
t_inicio = time.time()

# iterar el nombre y la url normal en un bucle para acceder
for nombre, url in categorias_data:
    cat_id = categorias_db[nombre]
    
    # llamar la funcion y almacenar los libros de cada categoria
    libros_de_categoria = scrapear_categoria(nombre, url, cat_id)
    
    # volcar los libros recolectados en la lista global
    libros_totales.extend(libros_de_categoria)

# inyeccion masiva y segura en sqlite
cur.executemany("""
    INSERT INTO books (title, price, rating, category_id, url)
    VALUES (?, ?, ?, ?, ?)
""", libros_totales)
conn.commit()

t_fin = time.time()
print(f"fase 3 completada: {len(libros_totales)} libros extraidos e insertados en {round(t_fin - t_inicio, 2)} segundos.")

iniciando extraccion concurrente con 10 hilos...
Fase 3 completada: 1000 libros extraidos e insertados en 10.4 segundos.


# La Prueba de API

In [ ]:
# sonda tactica de apis
titulo_prueba = "Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)"

#limpieza del titulo
titulo_limpio = re.sub(r'\(.*?\)', '', titulo_prueba).strip()

print("--- prueba de limpieza ---")
print(f"original: {titulo_prueba}")
print(f"limpio:   {titulo_limpio}\n")

print("--- conectando a open library ---")
url_busqueda = "https://openlibrary.org/search.json"
parametros = {
    "title": titulo_limpio, 
    "limit": 1, 
    "fields": "title,author_name,author_key"
}

try:
    respuesta_ol = requests.get(url_busqueda, params=parametros, timeout=10)
    datos_ol = respuesta_ol.json()
    
    if datos_ol.get("docs"):
        doc = datos_ol["docs"][0]
        nombre_autor = doc.get("author_name", ["desconocido"])[0]
        id_autor = doc.get("author_key", ["sin_id"])[0]
        
        print(f"autor encontrado: {nombre_autor}")
        print(f"id externo api:   {id_autor}")
        print("\njson devuelto por open library (fragmento):")
        print(doc)
    else:
        print("la api no encontro el libro, incluso limpio.")
        
except Exception as e:
    print(f"error en la sonda: {e}")

--- prueba de limpieza ---
original: Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)
limpio:   Princess Jellyfish 2-in-1 Omnibus, Vol. 01

--- conectando a open library ---
la api no encontro el libro, incluso limpio.


# Pipeline completo

In [ ]:
# ── pipeline secuencial
import re, requests, time

mapeo_paises = {
    "american": "USA", "british": "UK", "english": "UK", "scottish": "UK",
    "welsh": "UK", "irish": "Ireland", "french": "France", "spanish": "Spain",
    "german": "Germany", "italian": "Italy", "canadian": "Canada",
    "australian": "Australia", "japanese": "Japan", "russian": "Russia",
    "chinese": "China", "mexican": "Mexico", "argentine": "Argentina",
    "colombian": "Colombia", "chilean": "Chile", "brazilian": "Brazil",
    "indian": "India", "swedish": "Sweden", "norwegian": "Norway",
    "dutch": "Netherlands"
}

# identificador para la API, normalmente bloque las peticiones de python por seguridad
headers = {"User-Agent": "BooksScraperBot/1.0 (estudio_academico@ejemplo.com)"}
cache_autores = {}

# ─── fase 1 openlibrary
def buscar_en_openlibrary(book_id, titulo_original):
    """busca autor en open library de forma secuencial."""
    titulo_limpio = re.sub(r'\(.*?\)', '', titulo_original).strip()

    datos = {
        "name": "Unknown", "birth_year": None, "country": None,
        "external_api_id": None, "total_known_works": None,
        "api_source": "not_found"
    }

    try:
        # intentor con titulo limpio
        params = {"title": titulo_limpio, "limit": 1, "fields": "title,author_name,author_key"}
        res = requests.get("https://openlibrary.org/search.json", params=params, headers=headers, timeout=10)
        
        if not res.ok or not res.text.strip():
            return (book_id, datos)

        json_busqueda = res.json()

        # intento 2: sin subtitulo si fallo
        if not json_busqueda.get("docs") and ":" in titulo_limpio:
            params["title"] = titulo_limpio.split(":")[0].strip()
            res = requests.get("https://openlibrary.org/search.json", params=params, headers=headers, timeout=10)
            if res.ok and res.text.strip():
                json_busqueda = res.json()

        if not json_busqueda.get("docs"):
            return (book_id, datos)

        # consulta de author_key y name
        doc = json_busqueda["docs"][0]
        author_key  = doc.get("author_key",  [None])[0]
        author_name = doc.get("author_name", [None])[0]

        if not author_key or not author_name:
            return (book_id, datos)

        # cache, si ya se proceso este autor, reutilizar para ahorrar tiempo
        if author_key in cache_autores:
            return (book_id, cache_autores[author_key])

        datos["name"]            = author_name
        datos["external_api_id"] = author_key
        datos["api_source"]      = "open_library+wikipedia"

        # detalles del autor
        clean_key = author_key.replace("/authors/", "")
        res_autor = requests.get(f"https://openlibrary.org/authors/{clean_key}.json", headers=headers, timeout=10)

        #extraer fecha de nacimiento
        if res_autor.status_code == 200 and res_autor.text.strip():
            birth_raw = res_autor.json().get("birth_date", "")
            if birth_raw:
                digitos = [s for s in birth_raw.split() if s.isdigit() and len(s) == 4]
                datos["birth_year"] = int(digitos[0]) if digitos else None


        #consultar resto de sus obras
        res_obras = requests.get(f"https://openlibrary.org/authors/{clean_key}/works.json?limit=0", headers=headers, timeout=10)
        
        if res_obras.status_code == 200 and res_obras.text.strip():
            datos["total_known_works"] = res_obras.json().get("size", None)

        # guardar el resultado exitoso en el cache global
        cache_autores[author_key] = datos
        return (book_id, datos)

    except Exception as e:
        datos["api_source"] = f"error_ol: {str(e)[:20]}"
        return (book_id, datos)


# guardar todos los libros para procesarlos
cur.execute("SELECT id, title FROM books")
libros_db = cur.fetchall()
resultados_ol = []

print("── fase 1: open library secuencial")
t1 = time.time()

#bucle de procesar cada libro y buscar su autor
for i, libro in enumerate(libros_db, 1):
    resultado = buscar_en_openlibrary(libro[0], libro[1])
    resultados_ol.append(resultado)
    if i % 100 == 0:
        print(f"  {i}/1000 libros procesados...")

print(f"ok -> open library completado en {round(time.time()-t1, 2)}s")
encontrados = sum(1 for _, d in resultados_ol if d["external_api_id"])
print(f"  autores encontrados: {encontrados}/1000")


# ─── fase 2: wikipedia secuencial

print("\n── fase 2: wikipedia secuencial ───────")
t2 = time.time()

# filtro: solo los que tienen id oficial de open library y nombre real
# almacenar solo datos de autores que no se repiten para las consultas de pais
autores_unicos = {}
for book_id, datos in resultados_ol:
    key = datos.get("external_api_id")
    if key and key not in autores_unicos and datos["name"] != "Unknown":
        autores_unicos[key] = datos

print(f"  autores unicos a consultar en wikipedia: {len(autores_unicos)}")

#consultar los autores en wikipedia para obtener sus paises
for i, (author_key, datos) in enumerate(autores_unicos.items()):
    if i % 50 == 0:
        print(f"  {i}/{len(autores_unicos)} autores procesados...")

    try:
        # rest_v1 y quote para evitar errores de codificacion en las url
        # quote es para utilizar formato de web en espavios
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(datos['name'])}"
        res = requests.get(url, headers=headers, timeout=10)

        if res.status_code == 200 and res.text.strip():
            wiki_json = res.json()
            
            # extraer descripcion para buscar si hace referencia a un pais
            texto = (wiki_json.get("description", "") + " " + wiki_json.get("extract", "")).lower()

            for gentilicio, pais in mapeo_paises.items():
                if re.search(rf'\b{gentilicio}\b', texto):
                    datos["country"] = pais
                    # actualizacion de la memoria
                    cache_autores[author_key]["country"] = pais
                    break

    except Exception:
        pass

    # respeto al rate limit de la api
    time.sleep(0.8)  

print(f"ok -> wikipedia completado en {round(time.time()-t2, 2)}s")


# ─── fase 3: inyeccion segura en sqlite
print("\n── fase 3: inyectando en sqlite ────────────────────────────")

for book_id, info in resultados_ol:
    # inserta el autor, ignorando si el external_api_id ya existe en la base de datos
    cur.execute("""
        INSERT OR IGNORE INTO authors
            (name, birth_year, country, external_api_id, total_known_works, api_source)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (info["name"], info["birth_year"], info["country"],
          info["external_api_id"], info["total_known_works"], info["api_source"]))

    # recuperar el id local generado para la relacion del libro
    if info["external_api_id"]:
        cur.execute("SELECT id FROM authors WHERE external_api_id = ?", (info["external_api_id"],))
    else:
        cur.execute("SELECT id FROM authors WHERE name = ?", (info["name"],))

    fila = cur.fetchone()
    if fila:
        cur.execute("INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)", (book_id, fila[0]))

conn.commit()

# recoleccion de metricas
cur.execute("SELECT COUNT(*) FROM authors")
total = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM authors WHERE country IS NOT NULL")
con_pais = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM authors WHERE external_api_id IS NULL")
no_encontrados = cur.fetchone()[0]

print(f"\n✅ pipeline completo y persistido")
print(f"  total autores:              {total}")
print(f"  con pais:                   {con_pais}")
print(f"  no encontrados en api:      {no_encontrados}")
print(f"  % no encontrados:           {no_encontrados/len(libros_db)*100:.1f}%")

── fase 1: open library con 10 hilos ───────────────────────
  100/1000 libros procesados...
  200/1000 libros procesados...
  300/1000 libros procesados...
  400/1000 libros procesados...
  500/1000 libros procesados...
  600/1000 libros procesados...
  700/1000 libros procesados...
  800/1000 libros procesados...
  900/1000 libros procesados...
  1000/1000 libros procesados...
ok -> open library completado en 957.72s
  autores encontrados: 937/1000

── fase 2: wikipedia secuencial (tiempo estimado ~5-15 min) ───────
  autores unicos a consultar en wikipedia: 790
  0/790 autores procesados...
  50/790 autores procesados...
  100/790 autores procesados...
  150/790 autores procesados...
  200/790 autores procesados...
  250/790 autores procesados...
  300/790 autores procesados...
  350/790 autores procesados...
  400/790 autores procesados...
  450/790 autores procesados...
  500/790 autores procesados...
  550/790 autores procesados...
  600/790 autores procesados...
  650/790 autore

In [ ]:
# ── consultas de sql
import sqlite3
import time

conn = sqlite3.connect("books_pipeline.db")
cur = conn.cursor()

def imprimir_consulta(titulo, query):
    """funcion auxiliar para imprimir los resultados"""
    print(f"\n{titulo}")
    print("─" * 60)
    cur.execute(query)
    for fila in cur.fetchall():
        print(fila)

# libros con mas de 3 estrellas por menos de 10 lo que sea esa moneda
imprimir_consulta(
    "1. gangas de alta calidad (rating > 3 y precio < 10)",
    """
    SELECT title, price, rating 
    FROM books 
    WHERE rating > 3 AND price < 10.0 
    ORDER BY price ASC 
    LIMIT 5;
    """
)

# autor con peor promedio de rating (minimo 5 libros para evitar falsos positivos)
imprimir_consulta(
    "2. peores autores (peor promedio de rating, min 5 obras publicadas)",
    """
    SELECT a.name, ROUND(AVG(b.rating), 2) as avg_rating, COUNT(b.id) as total_books 
    FROM authors a 
    JOIN book_author ba ON a.id = ba.author_id 
    JOIN books b ON ba.book_id = b.id 
    GROUP BY a.id 
    HAVING total_books >= 5 
    ORDER BY avg_rating ASC 
    LIMIT 5;
    """
)

# categoria con mayor precio promedio
imprimir_consulta(
    "3. categorias mas caras (mayor precio promedio)",
    """
    SELECT c.name, ROUND(AVG(b.price), 2) as avg_price 
    FROM categories c 
    JOIN books b ON c.id = b.category_id 
    GROUP BY c.id 
    ORDER BY avg_price DESC 
    LIMIT 5;
    """
)

# top 5 autores con mas libros en la tienda
imprimir_consulta(
    "4. autores mas prolificos (mayor cantidad de libros)",
    """
    SELECT a.name, COUNT(b.id) as total_books 
    FROM authors a 
    JOIN book_author ba ON a.id = ba.author_id 
    JOIN books b ON ba.book_id = b.id 
    GROUP BY a.id 
    ORDER BY total_books DESC 
    LIMIT 5;
    """
)

# pais que produce mas libros con rating > 3
imprimir_consulta(
    "5. [obligatoria] top paises productores de literatura de alta calidad (rating > 3)",
    """
    SELECT a.country, COUNT(b.id) as high_rated_books 
    FROM authors a 
    JOIN book_author ba ON a.id = ba.author_id 
    JOIN books b ON ba.book_id = b.id 
    WHERE b.rating > 3 AND a.country IS NOT NULL 
    GROUP BY a.country 
    ORDER BY high_rated_books DESC 
    LIMIT 5;
    """
)

# ── prueba de indexacion y performance
print("\n── analisis de performance (busqueda de texto) ─────────────────────────")

# consulta sin indice
query_lenta = "SELECT title, price FROM books WHERE title LIKE '%Princess%'"

# medir sin indice 
t_inicio_lento = time.time()
for _ in range(100):
    cur.execute(query_lenta)
    cur.fetchall()
t_fin_lento = time.time()
tiempo_sin_indice = t_fin_lento - t_inicio_lento

print(f"tiempo de ejecucion sin indice (100 iteraciones): {tiempo_sin_indice:.5f} segundos")

# creacion del indice
print("inyectando indice en la columna 'title'...")
cur.execute("CREATE INDEX IF NOT EXISTS idx_books_title ON books(title);")
conn.commit()

# medir con indice
t_inicio_rapido = time.time()
for _ in range(100):
    cur.execute(query_lenta)
    cur.fetchall()
t_fin_rapido = time.time()
tiempo_con_indice = t_fin_rapido - t_inicio_rapido

print(f"Teiempo de ejecucion con indice (100 iteraciones): {tiempo_con_indice:.5f} segundos")

# conclusion 
mejora = ((tiempo_sin_indice - tiempo_con_indice) / tiempo_sin_indice) * 100 if tiempo_sin_indice > 0 else 0
print(f" conclusion: la indexacion mejoro la velocidad de busqueda en un {mejora:.1f}%")

conn.close()


1. gangas de alta calidad (rating > 3 y precio < 10)
────────────────────────────────────────────────────────────

2. peores autores (peor promedio de rating, min 5 obras publicadas)
────────────────────────────────────────────────────────────
('Sophie Kinsella', 2.33, 6)
('Worth Books', 2.33, 9)
('Cassandra Clare', 2.86, 7)
('Unknown', 2.89, 63)
('J.K. Rowling', 2.89, 9)

3. categorias mas caras (mayor precio promedio)
────────────────────────────────────────────────────────────
('Suspense', 58.33)
('Novels', 54.81)
('Politics', 53.61)
('Health', 51.45)
('New Adult', 46.38)

4. autores mas prolificos (mayor cantidad de libros)
────────────────────────────────────────────────────────────
('Unknown', 63)
('Stephen King', 12)
('J.K. Rowling', 9)
('Worth Books', 9)
('高屋奈月', 8)

5. [obligatoria] top paises productores de literatura de alta calidad (rating > 3)
────────────────────────────────────────────────────────────
('USA', 175)
('UK', 50)
('Canada', 8)
('Germany', 4)
('Australia', 3)